# Laboratorium 5: Ekstrakcja informacji -- reguły, NER i LLM

W tym laboratorium porównujemy trzy sposoby rozpoznawania nazw własnych (encji nazwanych, ang. named entity recognition, NER) w tekstach polskich:

`reguły → model NER → LLM z odpowiedzią w formacie JSON`

Wszystkie systemy działają na tym samym zbiorze referencyjnym i zwracają wyniki w tym samym formacie:

```json
{"text": "Olga Tokarczuk", "label": "PER", "start": 0, "end": 14}
```

Przyjmujemy konwencję `text[start:end]`: `start` jest indeksem pierwszego znaku encji, a `end` -- indeksem pierwszego znaku po niej (indeks wyłączny).

Główne pytanie laboratorium brzmi:

**Kiedy wystarczą reguły, kiedy lepszy jest model NER, a kiedy warto użyć LLM?**

## Po tych zajęciach student potrafi

- przygotować i skontrolować zbiór referencyjny oraz podzielić go na część rozwojową (DEV) i testową (TEST);
- zaimplementować ewaluację ścisłego i częściowego dopasowania zakresów;
- rozróżnić błąd klasy encji od błędu jej granic;
- zbudować prosty system regułowy;
- wykorzystać słownik nazw (`gazetteer`) w systemie regułowym i uwzględnić w nim fleksję języka polskiego;
- uruchomić i ocenić wstępnie wytrenowany model NER;
- sprawdzić wpływ progu pewności na precision i recall;
- uzyskać od LLM odpowiedź w formacie JSON i kontrolować jej poprawność;
- porównać jakość, szybkość i typy błędów trzech podejść;
- uzasadnić wybór rozwiązania dla konkretnego zastosowania.

## Ustawienia wstępne

Wymagane biblioteki:

```bash
pip install pandas numpy matplotlib transformers torch requests scikit-learn
```

**Model NER (język polski):**

```text
pczarnik/herbert-base-ner
```

To model HerBERT (`allegro/herbert-base-cased`) dostrojony do rozpoznawania nazw własnych na polskiej części zbioru WikiANN. Rozpoznaje klasy `PER`, `ORG` i `LOC`; etykiety mają postać IOB2 (`B-PER`, `I-PER` itd.). Pierwsze uruchomienie pobiera ok. 0,5 GB.

**Model językowy (LLM):** lokalnie przez [Ollamę](https://ollama.com). Model musi dobrze radzić sobie z językiem polskim. Propozycja:

```bash
ollama run hf.co/speakleash/Bielik-4.5B-v3.0-Instruct-GGUF:Q8_0
```

(polski model Bielik, ok. 5 GB, licencja Apache 2.0). Można użyć innego modelu wskazanego na zajęciach; w karcie systemu należy podać jego nazwę, kwantyzację i ustawienia generowania (`temperature`).

**Zasady oceny.** Wszystkie podejścia porównujemy na tym samym zbiorze referencyjnym. Zbiór dzielimy na część rozwojową (DEV) i testową (TEST):

- reguły, słownik i prompt tworzymy i poprawiamy **wyłącznie na DEV**;
- wszystkie tabele z wynikami liczymy **na TEST**, chyba że w poleceniu napisano inaczej;
- nie oceniamy systemów na podstawie pojedynczych przykładów.

In [ ]:
from pathlib import Path
import json
import re
import time
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import requests
from transformers import pipeline

ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)

pd.set_option("display.max_colwidth", 220)

## Zbiór referencyjny

Poniższy zbiór zawiera 30 polskich zdań z ręcznie oznaczonymi encjami:

- `PER`: osoba;
- `ORG`: organizacja;
- `LOC`: lokalizacja.

Ten sam zbiór będzie używany do oceny wszystkich trzech podejść. Dzielimy go na część rozwojową (DEV, 10 dokumentów) i testową (TEST, 20 dokumentów); podział jest zdefiniowany w komórce poniżej i nie wolno go zmieniać.

In [ ]:
RAW = [
("Donald Tusk spotkał się w Warszawie z przedstawicielami Komisji Europejskiej.",
 [("Donald Tusk","PER"),("Warszawie","LOC"),("Komisji Europejskiej","ORG")]),
("Olga Tokarczuk odebrała nagrodę podczas wydarzenia organizowanego przez Uniwersytet Wrocławski.",
 [("Olga Tokarczuk","PER"),("Uniwersytet Wrocławski","ORG")]),
("Robert Lewandowski przyleciał do Krakowa na spotkanie z przedstawicielami PZPN.",
 [("Robert Lewandowski","PER"),("Krakowa","LOC"),("PZPN","ORG")]),
("Microsoft otworzył nowe biuro w Warszawie, a Satya Nadella zapowiedział dalsze inwestycje.",
 [("Microsoft","ORG"),("Warszawie","LOC"),("Satya Nadella","PER")]),
("AGH w Krakowie podpisała umowę z firmą IBM.",
 [("AGH","ORG"),("Krakowie","LOC"),("IBM","ORG")]),
("Andrzej Sapkowski spotkał się z czytelnikami w Łodzi.",
 [("Andrzej Sapkowski","PER"),("Łodzi","LOC")]),
("Ministerstwo Finansów opublikowało raport przygotowany wspólnie z Narodowym Bankiem Polskim.",
 [("Ministerstwo Finansów","ORG"),("Narodowym Bankiem Polskim","ORG")]),
("Maria Nowak z firmy Allegro przeprowadziła prezentację w Poznaniu.",
 [("Maria Nowak","PER"),("Allegro","ORG"),("Poznaniu","LOC")]),
("Google zaprosił Jana Kowalskiego do swojego biura w Londynie.",
 [("Google","ORG"),("Jana Kowalskiego","PER"),("Londynie","LOC")]),
("Muzeum Narodowe w Krakowie otworzyło wystawę poświęconą Stanisławowi Wyspiańskiemu.",
 [("Muzeum Narodowe w Krakowie","ORG"),("Stanisławowi Wyspiańskiemu","PER")]),
("Joanna Bator była gościem Uniwersytetu Jagiellońskiego.",
 [("Joanna Bator","PER"),("Uniwersytetu Jagiellońskiego","ORG")]),
("Amazon rozwija centrum logistyczne pod Wrocławiem.",
 [("Amazon","ORG"),("Wrocławiem","LOC")]),
("Prezydent Warszawy Rafał Trzaskowski odwiedził Pragę.",
 [("Warszawy","LOC"),("Rafał Trzaskowski","PER"),("Pragę","LOC")]),
("Naukowcy z Politechniki Gdańskiej współpracują z zespołem z Uniwersytetu Stanforda.",
 [("Politechniki Gdańskiej","ORG"),("Uniwersytetu Stanforda","ORG")]),
("Anna Dymna prowadzi fundację Mimo Wszystko w Krakowie.",
 [("Anna Dymna","PER"),("Mimo Wszystko","ORG"),("Krakowie","LOC")]),
("Spotify zatrudnił Piotra Zielińskiego w swoim oddziale w Sztokholmie.",
 [("Spotify","ORG"),("Piotra Zielińskiego","PER"),("Sztokholmie","LOC")]),
("Bank Pekao współpracuje z Uniwersytetem Ekonomicznym w Katowicach.",
 [("Bank Pekao","ORG"),("Uniwersytetem Ekonomicznym w Katowicach","ORG")]),
("Marta Lempart uczestniczyła w konferencji zorganizowanej w Berlinie.",
 [("Marta Lempart","PER"),("Berlinie","LOC")]),
("Firma CD Projekt zaprosiła Adama Badowskiego na wydarzenie w Los Angeles.",
 [("CD Projekt","ORG"),("Adama Badowskiego","PER"),("Los Angeles","LOC")]),
("Europejska Agencja Kosmiczna współpracuje z NASA przy projekcie realizowanym w Paryżu.",
 [("Europejska Agencja Kosmiczna","ORG"),("NASA","ORG"),("Paryżu","LOC")]),
("Wisława Szymborska przez wiele lat mieszkała w Krakowie.",
 [("Wisława Szymborska","PER"),("Krakowie","LOC")]),
("Orange Polska otworzył punkt badawczy na Politechnice Warszawskiej.",
 [("Orange Polska","ORG"),("Politechnice Warszawskiej","ORG")]),
("Adam Małysz odwiedził Zakopane podczas wydarzenia organizowanego przez TVP Sport.",
 [("Adam Małysz","PER"),("Zakopane","LOC"),("TVP Sport","ORG")]),
("PKP Intercity uruchomiło nowe połączenie między Warszawą a Pragą.",
 [("PKP Intercity","ORG"),("Warszawą","LOC"),("Pragą","LOC")]),
("Michał Rusinek wystąpił na Uniwersytecie Śląskim w Katowicach.",
 [("Michał Rusinek","PER"),("Uniwersytecie Śląskim w Katowicach","ORG")]),
("Samsung zorganizował konferencję w Seulu, na której wystąpił Lee Jae-yong.",
 [("Samsung","ORG"),("Seulu","LOC"),("Lee Jae-yong","PER")]),
("Fundacja Batorego zaprosiła Anne Applebaum na debatę w Warszawie.",
 [("Fundacja Batorego","ORG"),("Anne Applebaum","PER"),("Warszawie","LOC")]),
("Grupa Azoty podpisała porozumienie z Akademią Górniczo-Hutniczą.",
 [("Grupa Azoty","ORG"),("Akademią Górniczo-Hutniczą","ORG")]),
("Iga Świątek wróciła z Paryża po turnieju organizowanym przez Francuską Federację Tenisową.",
 [("Iga Świątek","PER"),("Paryża","LOC"),("Francuską Federację Tenisową","ORG")]),
("Zespół OpenAI spotkał się z badaczami z MIT w Bostonie.",
 [("OpenAI","ORG"),("MIT","ORG"),("Bostonie","LOC")]),
]

def build_gold(raw):
    docs=[]
    for i,(text, ents) in enumerate(raw):
        out=[]
        used=0
        for entity,label in ents:
            start=text.find(entity, used)
            if start < 0:
                start=text.find(entity)
            if start < 0:
                raise ValueError((entity,text))
            end=start+len(entity)
            out.append({"text":entity,"label":label,"start":start,"end":end})
            used=end
        docs.append({"doc_id":i,"text":text,"entities":out})
    return docs

GOLD = build_gold(RAW)

# Podział zbioru: DEV (do budowy reguł, słownika i promptu); TEST (do oceny).

DEV  = [d for d in GOLD if d["doc_id"] % 3 == 0]
TEST = [d for d in GOLD if d["doc_id"] % 3 != 0]

print("dokumenty (razem / DEV / TEST):", len(GOLD), len(DEV), len(TEST))
print("encje (razem):", sum(len(d["entities"]) for d in GOLD))
print("encje w TEST:", Counter(e["label"] for d in TEST for e in d["entities"]))


# Zadanie 1: Audyt zbioru referencyjnego

Proszę zaimplementować kontrolę danych, która sprawdza:

1. czy `text[start:end]` jest identyczne z polem `text` encji;
2. czy `start < end`;
3. czy zakres mieści się w dokumencie;
4. czy etykieta należy do `{PER, ORG, LOC}`;
5. czy w jednym dokumencie nie występują zduplikowane adnotacje;
6. czy zakresy encji w jednym dokumencie nie nakładają się na siebie (przyjmujemy płaskie encje, bez zagnieżdżeń).

Następnie proszę podać:

- liczbę dokumentów;
- liczbę encji;
- rozkład klas (osobno dla DEV i TEST);
- średnią liczbę encji na dokument.

Na końcu proszę zapisać krótkie wytyczne adnotacji: co dokładnie uznajemy za `PER`, `ORG` i `LOC` oraz jak ustalamy granice encji. Wytyczne muszą rozstrzygać co najmniej trzy przypadki graniczne występujące w zbiorze, np.:

- czy dopełnienie lokalizacyjne należy do nazwy organizacji („Muzeum Narodowe w Krakowie” a „AGH w Krakowie”);
- czy stanowisko lub tytuł należy do encji `PER` („Prezydent Warszawy Rafał Trzaskowski”);
- czy nazwę miasta lub kraju wewnątrz nazwy instytucji oznaczamy dodatkowo jako `LOC`.

In [ ]:
# Proszę zaimplementować audyt zbioru referencyjnego.

# Zadanie 2: Metryki zakresowe i błędy granic

Proszę zaimplementować dwie procedury ewaluacyjne.

## Ścisłe dopasowanie (`exact span match`)

Predykcja jest poprawna tylko wtedy, gdy zgadzają się:

- `start`;
- `end`;
- `label`.

## Częściowe dopasowanie (`relaxed span match`)

Predykcja może zostać uznana za trafioną, jeżeli:

- zakres predykcji nachodzi na zakres referencyjny;
- dopasowanie jest wzajemnie jednoznaczne (1:1): każda predykcja i każda encja referencyjna należy do najwyżej jednej pary. Jeśli kilka predykcji nachodzi na jedną encję, dopasowujemy tę o największej części wspólnej; pozostałe liczymy jako fałszywie dodatnie.

Dla obu wariantów proszę policzyć precision, recall i F1. Metryki liczymy zbiorczo (micro) po wszystkich dokumentach; jeśli mianownik jest równy zero, przyjmujemy wartość 0.0.

Dodatkowo proszę zaimplementować liczenie błędów granic (`boundary error`): to para dopasowana w trybie częściowym, ale nie w ścisłym -- etykieta jest poprawna, zakresy nachodzą na siebie, ale `start` lub `end` się różnią.

Proszę przygotować co najmniej pięć ręcznie zapisanych przypadków testowych sprawdzających poprawność implementacji. Powinny one obejmować m.in.: idealną zgodność, błąd granicy, poprawny zakres z błędną klasą, dwie predykcje nachodzące na jedną encję, jedną predykcję obejmującą dwie encje oraz pustą listę predykcji.

In [ ]:
# Proszę zaimplementować:
# exact_span_metrics(...)
# relaxed_span_metrics(...)
# count_boundary_errors(...)


# Zadanie 3: System regułowy

Proszę zbudować pierwszy system ekstrakcji bez modelu uczącego się.

System powinien korzystać co najmniej z:

- wyrażeń regularnych;
- reguł dotyczących wielkich liter i sąsiedztwa słów;
- słownika nazw (`gazetteer`) zawierającego wybrane organizacje i lokalizacje.

Reguły, słownik i wyrażenia regularne tworzymy na podstawie części DEV oraz wiedzy ogólnej (np. listy największych polskich miast ustalonej **przed** obejrzeniem TEST). Słownik nie może zawierać nazw przepisanych ze zdań TEST.

Wskazówki dotyczące języka polskiego:

- w wyrażeniach regularnych wielką literę należy opisać także dla znaków diakrytycznych, np. `[A-ZĄĆĘŁŃÓŚŹŻ][a-ząćęłńóśźż]+` (klasa `[A-Z]` ich nie obejmuje);
- język polski jest fleksyjny: nazwa ze słownika („Kraków”) pojawia się w tekście w różnych formach („w Krakowie”, „do Krakowa”). Proszę zaproponować i zaimplementować sposób dopasowania odmienionych form (np. dopasowanie rdzenia lub prefiksu albo lista form) oraz opisać jego ograniczenie.

Wynik musi mieć ten sam format co zbiór referencyjny.

Proszę zmierzyć:

- exact precision / recall / F1;
- relaxed precision / recall / F1;
- liczbę błędów granic;
- F1 osobno dla `PER`, `ORG`, `LOC`;
- średni czas przetwarzania dokumentu.

Wyniki proszę podać osobno dla DEV i TEST i skomentować różnicę (jej duża wartość to sygnał dopasowania reguł do DEV). Proszę też krótko wskazać (na podstawie DEV), które reguły i które elementy słownika dały największą poprawę względem najprostszej wersji systemu.

In [ ]:
# Proszę zaimplementować system regułowy ze słownikiem encji.


# Zadanie 4: Model NER dla języka polskiego

Proszę uruchomić model:

```text
pczarnik/herbert-base-ner
```

Jest to model HerBERT (`allegro/herbert-base-cased`) dostrojony do rozpoznawania nazw własnych na polskiej części zbioru WikiANN. Zbiór WikiANN ma klasy `PER`, `ORG`, `LOC`, a jego adnotacje zostały wygenerowane automatycznie na podstawie Wikipedii (nie oznaczył ich ręcznie człowiek), więc zawierają szum.

**Wskazówka.** Wywołanie `pipeline("ner", model=..., aggregation_strategy="simple")` łączy podsłowa (tokeny podwyrazowe) w całe encje i zwraca ich zakresy znakowe `start`/`end` oraz wynik (`score`). Bez agregacji model zwraca wyniki dla pojedynczych podsłów.

Należy samodzielnie zaimplementować:

- przetwarzanie wszystkich dokumentów;
- zamianę etykiet `B-PER`, `I-PER` itd. na `PER`, `ORG`, `LOC` (i odrzucenie ewentualnych innych etykiet);
- zapis zakresów w formacie przyjętym w notatniku, wraz z polem `score`;
- pomiar czasu wnioskowania (z pominięciem pierwszego wywołania, które obejmuje inicjalizację modelu).

Proszę policzyć ten sam zestaw metryk co dla systemu regułowego (na TEST) i przedstawić oba systemy w jednej tabeli.

Proszę pamiętać, że model uczył się na zdaniach z Wikipedii, a nasz zbiór składa się ze zdań o znanych osobach i instytucjach. Wyniki mogą więc być zawyżone w porównaniu z tekstami z innych dziedzin (np. prasa, pisma urzędowe).

## Część opcjonalna: model spaCy

Proszę powtórzyć ewaluację dla modelu `pl_core_news_lg` z biblioteki spaCy (`python -m spacy download pl_core_news_lg`) i dodać go do tabeli. Etykiety spaCy należy odwzorować na naszą taksonomię:

- `persName` → `PER`;
- `orgName` → `ORG`;
- `placeName` i `geogName` → `LOC`;
- pozostałe etykiety (np. `date`, `time`) pomijamy.

Zakresy znakowe dostępne są jako `ent.start_char` i `ent.end_char`.

Proszę zwrócić uwagę, że:

- model spaCy został wytrenowany na innym zbiorze i według innych wytycznych adnotacji niż HerBERT, więc różnica wyników wynika nie tylko z jakości modelu, ale także z różnic w tym, co uznano za encję;
- spaCy nie zwraca wyniku (`score`) dla encji, więc model ten nie bierze udziału w zadaniu 5.

In [ ]:
# Proszę zaimplementować obsługę modelu NER i jego ewaluację.

# Zadanie 5: Próg pewności modelu NER

Proszę sprawdzić progi (dla modeu HerBERT):

```text
0.00 (bez progu), 0.50, 0.70, 0.90
```

Predykcje o wyniku poniżej progu należy odrzucać. Wynik encji jest średnią wyników jej podsłów (przy `aggregation_strategy="simple"`).

Dla każdego progu proszę policzyć:

- precision;
- recall;
- F1;
- coverage, czyli udział predykcji pozostających po zastosowaniu progu, liczony względem liczby predykcji bez progu (0.00).

Proszę przygotować jeden wykres pokazujący zależność precision i recall od progu.

Nie zakładamy, że wynik (`score`) modelu jest dobrze skalibrowaną miarą pewności; traktujemy go jako sygnał do eksperymentu z odrzucaniem predykcji. Modele dostrojone do rozpoznawania nazw bywają bardzo pewne swoich predykcji, więc jeśli wszystkie progi dają ten sam wynik, proszę dodać progi `0.95` i `0.99` i omówić tę obserwację -- brak zmian również jest wynikiem.

Eksperyment wykonujemy na całym zbiorze (DEV + TEST), ponieważ model nie był tu dostrajany. Jeśli jednak wybieramy próg do dalszego użycia (zadanie 7), wybieramy go na DEV, a jego wyniki raportujemy na TEST.

In [ ]:
# Proszę przeprowadzić eksperyment z progami pewności.


# Zadanie 6: LLM i odpowiedź w formacie JSON

Proszę przygotować polecenie dla modelu językowego (prompt), które nakazuje zwrócić wyłącznie obiekt JSON z listą encji, np. `{"entities": [{"text": "...", "label": "PER"}]}`. Model musi obsługiwać język polski (patrz „Ustawienia wstępne”). Prompt (w tym ewentualne przykłady wzorcowe) tworzymy na części DEV; przykłady z TEST nie mogą trafić do promptu. Zalecane ustawienie: `temperature = 0`.

Należy:

1. zdefiniować oczekiwany schemat odpowiedzi;
2. uruchomić LLM na całym zbiorze TEST;
3. sprawdzić poprawność składni JSON;
4. sprawdzić zgodność ze schematem;
5. odzyskać `start`/`end` na podstawie literalnego tekstu encji; jeśli ten sam tekst występuje w dokumencie kilka razy, każde jego wystąpienie przypisujemy najwyżej jednej encji;
6. wykrywać encje, których nie można odnaleźć w tekście;
7. zmierzyć średni czas przetwarzania dokumentu.

Dla odpowiedzi z niepoprawnym JSON-em proszę zastosować maksymalnie jedno automatyczne ponowne wywołanie (`retry`) z informacją o błędzie formatu.

Proszę policzyć:

- JSON validity (odsetek odpowiedzi będących poprawnym JSON-em);
- schema validity (odsetek odpowiedzi zgodnych ze schematem);
- exact F1;
- relaxed F1;
- F1 osobno dla każdej klasy;
- liczbę encji niewystępujących w tekście;
- średni czas na dokument;
- liczbę dodatkowych wywołań spowodowanych przez `retry`.

**Uwaga.** Ollama pozwala wymusić format odpowiedzi parametrem `format` (można w nim podać schemat JSON). Podstawowa wersja zadania ma polegać na samym poleceniu w prompcie, **bez** tego parametru -- wtedy poprawność formatu jest wynikiem eksperymentu, a `retry` ma sens.

## Część opcjonalna

Proszę powtórzyć eksperyment z parametrem `format` ze schematem JSON i porównać z wersją podstawową: poprawność formatu, jakość (exact F1) i czas.

In [ ]:
# Proszę zaimplementować wywołanie LLM,
# walidację struktury, odzyskiwanie zakresów i pojedynczy retry.


# Zadanie 7: Porównanie trzech podejść

Proszę przygotować jedną tabelę:

| system | exact F1 | relaxed F1 | boundary errors | PER F1 | ORG F1 | LOC F1 | ms/dok | poprawność formatu |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| reguły | | | | | | | | 1.00 |
| pretrained NER | | | | | | | | 1.00 |
| LLM | | | | | | | | |
| (opcjonalnie) model NER (spaCy) | | | | | | | | 1.00|



Proszę dodatkowo wskazać:

- system z najwyższym exact F1;
- system z najwyższym recall;
- system najszybszy;
- system o najbardziej przewidywalnym typie błędów.

Nie należy wybierać „zwycięzcy” na podstawie jednej liczby ani bez sprawdzenia, czy różnica jest większa niż niepewność oszacowania — zbiór TEST ma tylko 20 dokumentów i 52 encje.

*(opcjonalne)* Proszę podać 95% przedział ufności dla exact F1 każdego systemu (bootstrap po dokumentach, jak w laboratorium 2).

In [ ]:
# Proszę zebrać wyniki wszystkich systemów w jednej tabeli.


# Zadanie 8: Analiza błędów

Proszę przygotować wspólną tabelę dokładnie 12 błędnych przypadków ze zbioru TEST: po 4 z każdego z trzech systemów (reguły, HerBERT, LLM).

Proponowana taksonomia:

- `FP_SPURIOUS`: wykryto encję, której nie powinno być;
- `FN_MISSED`: pominięto encję;
- `BOUNDARY`: błędne granice;
- `LABEL`: błędna klasa;
- `NORMALIZATION`: problem normalizacji lub mapowania etykiety;
- `FORMAT`: niepoprawny format odpowiedzi LLM;
- `HALLUCINATION`: encja niewystępująca w tekście;
- `GOLD_ISSUE`: wątpliwa lub niespójna adnotacja w zbiorze referencyjnym
- `OTHER`: inny błąd.

Tabela:

| nr | system | doc_id | typ błędu | gold | predykcja | komentarz |
|---:|---|---:|---|---|---|---|

Proszę zadbać o to, aby tabela zawierała różne rodzaje błędów, a nie 12 wariantów tego samego problemu.


In [ ]:
# Proszę przygotować analizę błędnych przypadków.

# Zadanie 9: Interpretacja wyników

To zadanie nie wymaga nowej implementacji.

Na podstawie wyników proszę odpowiedzieć:

1. Dlaczego exact F1 i relaxed F1 mogą prowadzić do różnych ocen tego samego systemu?
2. W którym zastosowaniu błąd granic jest szczególnie poważny i dlaczego?
3. Jakie typy błędów dominują w systemie regułowym, modelu NER i LLM?
4. Czy wysoki próg modelu NER zawsze poprawia cały system? Proszę uzasadnić odpowiedź wynikami zadania 5.
5. Które ograniczenie obecnego zbioru referencyjnego najbardziej utrudnia wyciąganie ogólnych wniosków? (Proszę uwzględnić m.in. liczebność zbioru, dobór zdań oraz to, że modele mogły widzieć podobne nazwy podczas uczenia.)
6. Jak fleksja języka polskiego wpływa na działanie systemu regułowego ze słownikiem w porównaniu z modelem NER?

# Zadanie 10: Decyzja projektowa

To zadanie nie wymaga nowej implementacji.

Proszę wybrać rozwiązanie dla trzech scenariuszy:

1. anonimizacja danych osobowych, gdzie pominięcie encji jest bardzo kosztowne;
2. szybkie oznaczanie znanych organizacji i lokalizacji w dużym strumieniu tekstu;
3. system, w którym schemat encji może często się zmieniać.

Dla każdego scenariusza proszę podać:

- wybrane podejście;
- jedną metrykę uzasadniającą wybór;
- dominujący typ błędu;
- najważniejszy koszt lub ograniczenie.

Na końcu proszę odpowiedzieć jednym akapitem:

**Dlaczego model o najwyższym F1 nie musi być najlepszym systemem ekstrakcji informacji?**


# Karta systemu

Proszę wybrać rozwiązanie rekomendowane do jednego konkretnego zastosowania i utworzyć:

```text
artifacts/information_extraction_model_card.md
```

Karta powinna zawierać:

## System
- wybrane podejście: reguły / NER / LLM;
- nazwa modelu, źródło (np. Hugging Face lub Ollama), kwantyzacja lub wersja, licencja -- jeśli dotyczy;
- format zwracanych encji.

## Zadanie
- rozpoznawane klasy;
- wytyczne anotacji;
- konwencja zakresów.

## Dane
- liczba dokumentów i encji (osobno DEV i TEST);
- rozkład klas.

## Ewaluacja (na TEST)
- exact precision / recall / F1;
- relaxed precision / recall / F1;
- F1 per klasa;
- błędy granic.

## Koszt
- średni czas na dokument;
- dodatkowe wymagania modelowe lub infrastrukturalne.

## Błędy
- dominujące typy błędów z analizy 12 przypadków.

## Zastosowanie
- do czego system jest rekomendowany;
- kiedy należy wybrać inne podejście.

## Ograniczenia
- ograniczenia zbioru (m.in. liczebność, dobór zdań);
- ograniczenia metryk;
- ograniczenia samego systemu.

# Warunki zaliczenia laboratorium

Laboratorium jest ukończone, jeśli zawiera:

- [ ] audyt zbioru referencyjnego i podział na DEV / TEST;
- [ ] własną implementację exact precision / recall / F1;
- [ ] własną implementację relaxed precision / recall / F1;
- [ ] wykrywanie błędów granic;
- [ ] system regułowy ze słownikiem nazw (`gazetteer`) i obsługą fleksji;
- [ ] model NER dla języka polskiego (HerBERT);
- [ ] eksperyment z progami `0.00`, `0.50`, `0.70`, `0.90`;
- [ ] LLM (z obsługą języka polskiego) zwracający odpowiedź w formacie JSON;
- [ ] JSON validity i schema validity;
- [ ] odzyskiwanie `start/end`;
- [ ] maksymalnie jeden `retry` dla błędu formatu;
- [ ] wspólną tabelę porównawczą trzech podejść;
- [ ] tabelę 12 błędnych przypadków (po 4 z każdego systemu);
- [ ] odpowiedzi do zadań interpretacyjnych (9 i 10);
- [ ] plik `artifacts/information_extraction_model_card.md`.